# Data Wrangling: Schools

This notebook creates school access and school strength features for San Diego County census tracts.

For this layer, I’m using schools in two ways. First, I want to measure school access, like how many schools are in or near each tract. Second, I want to bring in school performance signals using public academic indicator data.

This won’t be a perfect school quality ranking system. School quality is more complicated than one score. But these features should help show which tracts might be stronger fits for family-oriented housing or development.

In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np

from pathlib import Path

In [2]:
# setting file paths
schools_path = Path('../data/raw/schools/sd_schools_2024.csv')
districts_path = Path('../data/raw/schools/DistrictAreas2425/DistrictAreas2425.shp')
ela_path = Path('../data/raw/schools/academic_indicators_ela_24-25.xlsx')
math_path = Path('../data/raw/schools/academic_indictors_math_24-25.xlsx')

In [3]:
# schools dataset
schools = pd.read_csv(schools_path)

# district boundaries
districts = gpd.read_file(districts_path)

# academic indicators
ela = pd.read_excel(ela_path)
math = pd.read_excel(math_path)

In [4]:
# checking that everything loaded
print('schools:', schools.shape)
print('districts:', districts.shape)
print('ela:', ela.shape)
print('math:', math.shape)

schools: (9982, 98)
districts: (937, 54)
ela: (176088, 37)
math: (176260, 37)


In [5]:
schools.columns

Index(['OBJECTID', 'Academic Year', 'Fed ID', 'CDS Code', 'District Code',
       'School Code', 'Region', 'County Name', 'District Name', 'School Name',
       'School Type', 'Status', 'Open Date', 'Closed Date', 'School Level',
       'Grade Low', 'Grade High', 'Charter', 'Charter Num', 'Funding Type',
       'Virtual', 'Magnet', 'Title I', 'DASS', 'Assistance Status ESSA',
       'Street', 'City', 'Zip', 'State', 'Locale', 'School Website',
       'Enroll Total', 'African American', 'African American (%)',
       'American Indian', 'American Indian Pct', 'Asian', 'Asian (%)',
       'Filipino', 'Filipino (%)', 'Hispanic', 'Hispanic (%)',
       'Pacific Islander', 'Pacific Islander (%)', 'White', 'White (%)',
       'Two or More Races', 'Two or More Races (%)', 'Not Reported',
       'Not Reported (%)', 'English Learner', 'English Learner (%)', 'Foster',
       'Foster (%)', 'Homeless', 'Homeless (%)', 'Migrant', 'Migrant (%)',
       'Socioeconomically Disadvantaged',
       'Socio

In [6]:
districts.columns

Index(['OBJECTID', 'Year', 'FedID', 'CDCode', 'CDSCode', 'CountyName',
       'DistrictNa', 'DistrictTy', 'GradeLow', 'GradeHigh', 'GradeLowCe',
       'GradeHighC', 'AssistStat', 'CongressUS', 'SenateCA', 'AssemblyCA',
       'UpdateNote', 'EnrollTota', 'EnrollChar', 'EnrollNonC', 'AAcount',
       'AApct', 'AIcount', 'AIpct', 'AScount', 'ASpct', 'FIcount', 'FIpct',
       'HIcount', 'HIpct', 'PIcount', 'PIpct', 'WHcount', 'WHpct', 'MRcount',
       'MRpct', 'NRcount', 'NRpct', 'ELcount', 'ELpct', 'FOScount', 'FOSpct',
       'HOMcount', 'HOMpct', 'MIGcount', 'MIGpct', 'SWDcount', 'SWDpct',
       'SEDcount', 'SEDpct', 'DistrctAre', 'Shape__Are', 'Shape__Len',
       'geometry'],
      dtype='str')

In [7]:
ela.columns

Index(['cds', 'rtype', 'schoolname', 'districtname', 'countyname',
       'charter_flag', 'coe_flag', 'dass_flag', 'studentgroup', 'currdenom',
       'currstatus', 'priordenom', 'priorstatus', 'change', 'statuslevel',
       'changelevel', 'color', 'box', 'currnsizemet', 'priornsizemet',
       'accountabilitymet', 'hscutpoints', 'pairshare_method',
       'currprate_enrolled', 'currprate_tested', 'currprate', 'currnumPRLOSS',
       'currdenom_withoutPRLOSS', 'currstatus_withoutPRLOSS',
       'priorprate_enrolled', 'priorprate_tested', 'priorprate',
       'priornumPRLOSS', 'priordenom_withoutPRLOSS',
       'priorstatus_withoutPRLOSS', 'indicator', 'reportingyear'],
      dtype='str')

In [8]:
math.columns

Index(['cds', 'rtype', 'schoolname', 'districtname', 'countyname',
       'charter_flag', 'coe_flag', 'dass_flag', 'studentgroup', 'currdenom',
       'currstatus', 'priordenom', 'priorstatus', 'change', 'statuslevel',
       'changelevel', 'color', 'box', 'currnsizemet', 'priornsizemet',
       'accountabilitymet', 'hscutpoints', 'pairshare_method',
       'currprate_enrolled', 'currprate_tested', 'currprate', 'currnumPRLOSS',
       'currdenom_withoutPRLOSS', 'currstatus_withoutPRLOSS',
       'priorprate_enrolled', 'priorprate_tested', 'priorprate',
       'priornumPRLOSS', 'priordenom_withoutPRLOSS',
       'priorstatus_withoutPRLOSS', 'indicator', 'reportingyear'],
      dtype='str')

In [9]:
schools.info()

<class 'pandas.DataFrame'>
RangeIndex: 9982 entries, 0 to 9981
Data columns (total 98 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   OBJECTID                             9982 non-null   int64  
 1   Academic Year                        9982 non-null   str    
 2   Fed ID                               9982 non-null   int64  
 3   CDS Code                             9982 non-null   int64  
 4   District Code                        9982 non-null   int64  
 5   School Code                          9982 non-null   int64  
 6   Region                               9982 non-null   int64  
 7   County Name                          9982 non-null   str    
 8   District Name                        9982 non-null   str    
 9   School Name                          9982 non-null   str    
 10  School Type                          9982 non-null   str    
 11  Status                               9982

In [10]:
schools.isna().sum().sort_values(ascending=False).head(30)

Closed Date               9932
Charter Num               8709
Funding Type              8709
DistrictHighNameGeo       7110
DistrictHighCodeGeo       7110
DistrictElemNameGeo       6934
DistrictElemCodeGeo       6934
DistrictUnifiedCodeGeo    2877
DistrictUnifiedNameGeo    2877
School Website            2717
DASS                        55
DASS2                       55
Title I                     51
Assistance Status ESSA      51
Grade High                  18
Magnet2                     10
Magnet                      10
School Level                 6
Latitude                     2
Longitude                    2
Charter                      0
Grade Low                    0
School Type                  0
County Name                  0
Status                       0
Open Date                    0
Academic Year                0
OBJECTID                     0
School Name                  0
Fed ID                       0
dtype: int64

In [11]:
schools['County Name'].value_counts().head(20)

County Name
Los Angeles       2154
San Diego          754
Orange             626
San Bernardino     565
Riverside          504
Santa Clara        403
Sacramento         386
Alameda            371
Fresno             351
Contra Costa       274
Kern               272
San Joaquin        243
Ventura            212
Tulare             193
Stanislaus         185
San Mateo          170
Sonoma             170
Monterey           133
Placer             128
San Francisco      124
Name: count, dtype: int64

In [12]:
# filtering to San Diego County and active schools
sd_schools = schools[
    (schools['County Name'] == 'San Diego') &
    (schools['Status'] == 'Active')].copy()

sd_schools.shape

(751, 98)

In [13]:
sd_schools.head()

,OBJECTID,Academic Year,Fed ID,CDS Code,District Code,School Code,Region,County Name,District Name,School Name,...,CongUSGeo,SenateCAGeo,AssemblyCAGeo,Locale Four,Virtual2,Charter2,Magnet2,DASS2,x,y
6783,6784,2024-25,69103011385,37103710108548,10371,108548,9,San Diego,San Diego County Office of Education,Iftin Charter,...,52,39,79,1 - City,Not Virtual,Yes,No,No,-1.303300e+07,3.863112e+06
6784,6785,2024-25,69103012159,37103710115998,10371,115998,9,San Diego,San Diego County Office of Education,San Pasqual Academy,...,48,40,75,4 - Rural,Primarily Classroom,No,No,Yes,-1.301859e+07,3.906891e+06
6785,6786,2024-25,69103012550,37103710120485,10371,120485,9,San Diego,San Diego County Office of Education,Davila Day,...,52,18,80,1 - City,Not Virtual,No,No,Yes,-1.303417e+07,3.846895e+06
6786,6787,2024-25,69103012496,37103710120493,10371,120493,9,San Diego,San Diego County Office of Education,Monarch,...,52,18,80,1 - City,Primarily Classroom,No,No,Yes,-1.304112e+07,3.855869e+06
6787,6788,2024-25,69103012866,37103710124321,10371,124321,9,San Diego,San Diego County Office of Education,Howard Gardner Community Charter,...,52,18,80,1 - City,Not Virtual,Yes,No,No,-1.303501e+07,3.847786e+06


## Check school fields

Now that the data is filtered to San Diego County, I’m checking the fields that describe each school.

I need to identify school type, grade span, charter status, district, and location fields before creating tract-level features.

In [14]:
# checking columns related to school type, grades, charter status, and location
school_cols = [
    col for col in sd_schools.columns
    if any(word in col.lower() for word in [
        'school', 'district', 'type', 'grade', 'charter',
        'lat', 'lon', 'long', 'x', 'y', 'address', 'city', 'zip'])]

school_cols

['Academic Year',
 'District Code',
 'School Code',
 'County Name',
 'District Name',
 'School Name',
 'School Type',
 'School Level',
 'Grade Low',
 'Grade High',
 'Charter',
 'Charter Num',
 'Funding Type',
 'City',
 'Zip',
 'School Website',
 'Socioeconomically Disadvantaged',
 'Socioeconomically Disadvantaged (%)',
 'Grade TK',
 'Grade KG',
 'Grade 1',
 'Grade 2',
 'Grade 3',
 'Grade 4',
 'Grade 5',
 'Grade 6',
 'Grade 7',
 'Grade 8',
 'Grade 9',
 'Grade 10',
 'Grade 11',
 'Grade 12',
 'Latitude',
 'Longitude',
 'CountyCodeGeo',
 'CountyNameGeo',
 'DistrictElemCodeGeo',
 'DistrictElemNameGeo',
 'DistrictHighCodeGeo',
 'DistrictHighNameGeo',
 'DistrictUnifiedCodeGeo',
 'DistrictUnifiedNameGeo',
 'AssemblyCAGeo',
 'Charter2',
 'x',
 'y']

In [15]:
# previewing the useful school fields
sd_schools['District Name'].value_counts()

District Name
San Diego Unified                               214
Chula Vista Elementary                           50
Poway Unified                                    39
Vista Unified                                    31
Sweetwater Union High                            30
Cajon Valley Union                               29
Escondido Union                                  26
Oceanside Unified                                25
La Mesa-Spring Valley                            22
San Marcos Unified                               20
San Diego County Office of Education             18
Grossmont Union High                             18
Carlsbad Unified                                 16
Mountain Empire Unified                          13
Lakeside Union Elementary                        11
National Elementary                              11
Ramona City Unified                              11
Santee                                           11
South Bay Union                                  1

In [16]:
# checking school level categories
sd_schools['School Level'].value_counts(dropna=False)

School Level
Elementary         454
High               138
Middle             102
Other               56
Adult Education      1
Name: count, dtype: int64

In [17]:
# checking school type categories
sd_schools['School Type'].value_counts(dropna=False)

School Type
Elementary                       445
Middle                            98
High                              93
K-12                              51
Alternative Schools of Choice     26
Continuation                      16
Special Education                 13
Community Day                      4
Juvenile Court                     2
County Community                   2
Opportunity                        1
Name: count, dtype: int64

In [18]:
# checking charter counts
sd_schools['Charter'].value_counts(dropna=False)

Charter
N    626
Y    125
Name: count, dtype: int64

In [19]:
# counting active schools by district
schools_by_district = (
    sd_schools
    .groupby('District Name')
    .size()
    .reset_index(name='school_count')
    .sort_values('school_count', ascending=False))

schools_by_district.head(20)

,District Name,school_count
35,San Diego Unified,214
6,Chula Vista Elementary,50
25,Poway Unified,39
47,Vista Unified,31
44,Sweetwater Union High,30
3,Cajon Valley Union,29
11,Escondido Union,26
24,Oceanside Unified,25
19,La Mesa-Spring Valley,22
37,San Marcos Unified,20


In [20]:
# counting school levels by district
schools_by_district_level = (
    sd_schools
    .groupby(['District Name', 'School Level'])
    .size()
    .reset_index(name='school_count')
    .sort_values(['District Name', 'school_count'], ascending=[True, False]))

schools_by_district_level.head(40)

,District Name,School Level,school_count
0,Alpine Union Elementary,Elementary,4
1,Alpine Union Elementary,Middle,1
2,Bonsall Unified,Elementary,3
3,Bonsall Unified,High,1
4,Bonsall Unified,Middle,1
6,Borrego Springs Unified,High,2
5,Borrego Springs Unified,Elementary,1
7,Borrego Springs Unified,Middle,1
8,Cajon Valley Union,Elementary,22
9,Cajon Valley Union,Middle,6


In [21]:
# making district counts easier to compare
district_level_pivot = (
    schools_by_district_level.pivot_table(
        index='District Name',
        columns='School Level',
        values='school_count',
        fill_value=0)
    .reset_index())

district_level_pivot.sort_values(by='Elementary', ascending=False)

School Level,District Name,Adult Education,Elementary,High,Middle,Other
35,San Diego Unified,0.0,137.0,37.0,28.0,12.0
6,Chula Vista Elementary,0.0,47.0,0.0,0.0,3.0
25,Poway Unified,0.0,26.0,7.0,6.0,0.0
3,Cajon Valley Union,0.0,22.0,0.0,6.0,1.0
11,Escondido Union,0.0,20.0,0.0,6.0,0.0
19,La Mesa-Spring Valley,0.0,18.0,0.0,4.0,0.0
47,Vista Unified,1.0,16.0,7.0,4.0,3.0
24,Oceanside Unified,0.0,14.0,4.0,4.0,3.0
37,San Marcos Unified,0.0,12.0,4.0,3.0,1.0
23,National Elementary,0.0,11.0,0.0,0.0,0.0


## Clean school level groups

The school level field has more detail than I need, so I’m grouping them into broader categories to count elementary, middle, high, and other school types by tract later.

In [22]:
# checking school level values before grouping
sd_schools['School Level'].value_counts(dropna=False)

School Level
Elementary         454
High               138
Middle             102
Other               56
Adult Education      1
Name: count, dtype: int64

In [23]:
# grouping school levels into simpler categories
def clean_school_level(level):
    if pd.isna(level):
        return 'other'
    elif 'Elementary' in level:
        return 'elementary'
    elif 'Middle' in level:
        return 'middle'
    elif 'High' in level:
        return 'high'
    else:
        return 'other'


sd_schools['school_level_clean'] = sd_schools['School Level'].apply(clean_school_level)

In [24]:
# checking cleaned school level counts
sd_schools['school_level_clean'].value_counts()

school_level_clean
elementary    454
high          138
middle        102
other          57
Name: count, dtype: int64

In [25]:
# counting cleaned school levels by district
district_level_clean = (
    sd_schools
    .groupby(['District Name', 'school_level_clean'])
    .size()
    .reset_index(name='school_count')
    .sort_values(['District Name', 'school_count'], ascending=[True, False]))

district_level_clean.head(40)

,District Name,school_level_clean,school_count
0,Alpine Union Elementary,elementary,4
1,Alpine Union Elementary,middle,1
2,Bonsall Unified,elementary,3
3,Bonsall Unified,high,1
4,Bonsall Unified,middle,1
6,Borrego Springs Unified,high,2
5,Borrego Springs Unified,elementary,1
7,Borrego Springs Unified,middle,1
8,Cajon Valley Union,elementary,22
9,Cajon Valley Union,middle,6


In [26]:
# making district school level counts easier to read
district_level_clean_pivot = (
    district_level_clean
    .pivot_table(
        index='District Name',
        columns='school_level_clean',
        values='school_count',
        fill_value=0)
    .reset_index())

district_level_clean_pivot.head(20)

school_level_clean,District Name,elementary,high,middle,other
0,Alpine Union Elementary,4.0,0.0,1.0,0.0
1,Bonsall Unified,3.0,1.0,1.0,0.0
2,Borrego Springs Unified,1.0,2.0,1.0,0.0
3,Cajon Valley Union,22.0,0.0,6.0,1.0
4,Cardiff Elementary,2.0,0.0,0.0,0.0
5,Carlsbad Unified,9.0,3.0,3.0,1.0
6,Chula Vista Elementary,47.0,0.0,0.0,3.0
7,Coronado Unified,2.0,1.0,1.0,0.0
8,Dehesa Elementary,2.0,0.0,0.0,3.0
9,Del Mar Union Elementary,9.0,0.0,0.0,0.0


## Clean charter schools

Converting charter schools to binary so it's easier to use later. 

In [27]:
# checking charter values before cleaning
sd_schools['Charter'].value_counts(dropna=False)

Charter
N    626
Y    125
Name: count, dtype: int64

In [28]:
# converting charter yes/no into a binary flag
sd_schools['charter_flag'] = sd_schools['Charter'].map({'Y': 1, 'N': 0})

In [29]:
# checking the new flag
sd_schools[['Charter', 'charter_flag']].value_counts(dropna=False)

Charter  charter_flag
N        0               626
Y        1               125
Name: count, dtype: int64

## Convert schools to points

Next I’m turning each school into a spatial point using longitude and latitude.

This lets me join schools to census tracts later.

In [30]:
# checking coordinate fields before creating points
sd_schools[['Latitude', 'Longitude']].isna().sum()

Latitude     0
Longitude    0
dtype: int64

In [31]:
# previewing school coordinates
sd_schools[['School Name', 'Latitude', 'Longitude']].head()

,School Name,Latitude,Longitude
6783,Iftin Charter,32.757134,-117.077443
6784,San Pasqual Academy,33.087253,-116.947991
6785,Davila Day,32.634541,-117.087983
6786,Monarch,32.702400,-117.150407
6787,Howard Gardner Community Charter,32.641281,-117.095509


In [32]:
# use longitude and latitude to make map points, and store it as spatial data so I can join it to census tracts later
sd_schools_gdf = gpd.GeoDataFrame(
    sd_schools,
    geometry=gpd.points_from_xy(sd_schools['Longitude'], sd_schools['Latitude']),
    crs='EPSG:4326')

sd_schools_gdf.head()

,OBJECTID,Academic Year,Fed ID,CDS Code,District Code,School Code,Region,County Name,District Name,School Name,...,Locale Four,Virtual2,Charter2,Magnet2,DASS2,x,y,school_level_clean,charter_flag,geometry
6783,6784,2024-25,69103011385,37103710108548,10371,108548,9,San Diego,San Diego County Office of Education,Iftin Charter,...,1 - City,Not Virtual,Yes,No,No,-1.303300e+07,3.863112e+06,elementary,1,POINT (-117.07744 32.75713)
6784,6785,2024-25,69103012159,37103710115998,10371,115998,9,San Diego,San Diego County Office of Education,San Pasqual Academy,...,4 - Rural,Primarily Classroom,No,No,Yes,-1.301859e+07,3.906891e+06,high,0,POINT (-116.94799 33.08725)
6785,6786,2024-25,69103012550,37103710120485,10371,120485,9,San Diego,San Diego County Office of Education,Davila Day,...,1 - City,Not Virtual,No,No,Yes,-1.303417e+07,3.846895e+06,elementary,0,POINT (-117.08798 32.63454)
6786,6787,2024-25,69103012496,37103710120493,10371,120493,9,San Diego,San Diego County Office of Education,Monarch,...,1 - City,Primarily Classroom,No,No,Yes,-1.304112e+07,3.855869e+06,other,0,POINT (-117.15041 32.7024)
6787,6788,2024-25,69103012866,37103710124321,10371,124321,9,San Diego,San Diego County Office of Education,Howard Gardner Community Charter,...,1 - City,Not Virtual,Yes,No,No,-1.303501e+07,3.847786e+06,elementary,1,POINT (-117.09551 32.64128)


In [33]:
# checking the school point CRS
sd_schools_gdf.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [34]:
# checking CRS for each spatial layer
print('schools:', sd_schools_gdf.crs)
print('districts:', districts.crs)

schools: EPSG:4326
districts: EPSG:3857


## Load census tracts

Loading the census tract boundaries so I can assign each school point to a tract.

In [35]:
# loading census tract boundaries
tracts_path = Path('../data/raw/census_tracts/tl_2024_06_tract.shp')

tracts = gpd.read_file(tracts_path)

tracts.head()

,STATEFP,COUNTYFP,TRACTCE,GEOID,GEOIDFQ,NAME,NAMELSAD,MTFCC,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry
0,06,001,442700,06001442700,1400000US06001442700,4427,Census Tract 4427,G5020,S,1234016,0,+37.5371513,-122.0081095,"POLYGON ((-122.01721 37.53932, -122.01719 37.5..."
1,06,001,442800,06001442800,1400000US06001442800,4428,Census Tract 4428,G5020,S,1278646,0,+37.5293619,-121.9931002,"POLYGON ((-122.0023 37.52984, -122.00224 37.52..."
2,06,037,204920,06037204920,1400000US06037204920,2049.20,Census Tract 2049.20,G5020,S,909972,0,+34.0175004,-118.1974975,"POLYGON ((-118.20284 34.01966, -118.20283 34.0..."
3,06,037,205110,06037205110,1400000US06037205110,2051.10,Census Tract 2051.10,G5020,S,286962,0,+34.0245059,-118.2142985,"POLYGON ((-118.21964 34.02628, -118.21945 34.0..."
4,06,037,205120,06037205120,1400000US06037205120,2051.20,Census Tract 2051.20,G5020,S,1466242,0,+34.0187542,-118.2117951,"POLYGON ((-118.22023 34.02056, -118.22018 34.0..."


In [36]:
# checking tract CRS
tracts.crs

<Geographic 2D CRS: EPSG:4269>
Name: NAD83
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: North America - onshore and offshore: Canada - Alberta; British Columbia; Manitoba; New Brunswick; Newfoundland and Labrador; Northwest Territories; Nova Scotia; Nunavut; Ontario; Prince Edward Island; Quebec; Saskatchewan; Yukon. Puerto Rico. United States (USA) - Alabama; Alaska; Arizona; Arkansas; California; Colorado; Connecticut; Delaware; Florida; Georgia; Hawaii; Idaho; Illinois; Indiana; Iowa; Kansas; Kentucky; Louisiana; Maine; Maryland; Massachusetts; Michigan; Minnesota; Mississippi; Missouri; Montana; Nebraska; Nevada; New Hampshire; New Jersey; New Mexico; New York; North Carolina; North Dakota; Ohio; Oklahoma; Oregon; Pennsylvania; Rhode Island; South Carolina; South Dakota; Tennessee; Texas; Utah; Vermont; Virginia; Washington; West Virginia; Wisconsin; Wyoming. US Virgin Islands. British Virgin Islands

In [37]:
# filtering to San Diego County tracts
sd_tracts = tracts[tracts['COUNTYFP'] == '073'].copy()

sd_tracts.shape

(737, 14)

In [38]:
# keeping useful tract fields
sd_tracts = sd_tracts[['GEOID', 'NAMELSAD', 'geometry']].copy()

sd_tracts.head()

,GEOID,NAMELSAD,geometry
817,06073008331,Census Tract 83.31,"POLYGON ((-117.23082 32.94176, -117.23079 32.9..."
818,06073008336,Census Tract 83.36,"POLYGON ((-117.13793 32.96927, -117.13792 32.9..."
819,06073008337,Census Tract 83.37,"POLYGON ((-117.14678 32.95497, -117.14657 32.9..."
820,06073011601,Census Tract 116.01,"POLYGON ((-117.10356 32.6672, -117.10314 32.66..."
821,06073011602,Census Tract 116.02,"POLYGON ((-117.10154 32.66202, -117.10133 32.6..."


In [39]:
# matching schools and districts to tract CRS
sd_schools_gdf = sd_schools_gdf.to_crs(sd_tracts.crs)
districts = districts.to_crs(sd_tracts.crs)

print('schools:', sd_schools_gdf.crs)
print('districts:', districts.crs)
print('tracts:', sd_tracts.crs)

schools: EPSG:4269
districts: EPSG:4269
tracts: EPSG:4269


## Join schools to census tracts

Now that schools and tracts use the same CRS, I can spatially join each school point to the tract it falls inside.

This gives each school a tract GEOID, which I’ll use to create tract-level school counts.

In [40]:
# joining each school point to a census tract
schools_tract_join = gpd.sjoin(
    sd_schools_gdf,
    sd_tracts,
    how='left',
    predicate='within') # only join the tract if the school point is inside the tract polygon

schools_tract_join.head()

,OBJECTID,Academic Year,Fed ID,CDS Code,District Code,School Code,Region,County Name,District Name,School Name,...,Magnet2,DASS2,x,y,school_level_clean,charter_flag,geometry,index_right,GEOID,NAMELSAD
6783,6784,2024-25,69103011385,37103710108548,10371,108548,9,San Diego,San Diego County Office of Education,Iftin Charter,...,No,No,-1.303300e+07,3.863112e+06,elementary,1,POINT (-117.07744 32.75713),6200.0,06073002702,Census Tract 27.02
6784,6785,2024-25,69103012159,37103710115998,10371,115998,9,San Diego,San Diego County Office of Education,San Pasqual Academy,...,No,Yes,-1.301859e+07,3.906891e+06,high,0,POINT (-116.94799 33.08725),5714.0,06073020801,Census Tract 208.01
6785,6786,2024-25,69103012550,37103710120485,10371,120485,9,San Diego,San Diego County Office of Education,Davila Day,...,No,Yes,-1.303417e+07,3.846895e+06,elementary,0,POINT (-117.08798 32.63454),6765.0,06073012700,Census Tract 127
6786,6787,2024-25,69103012496,37103710120493,10371,120493,9,San Diego,San Diego County Office of Education,Monarch,...,No,Yes,-1.304112e+07,3.855869e+06,other,0,POINT (-117.15041 32.7024),2112.0,06073005103,Census Tract 51.03
6787,6788,2024-25,69103012866,37103710124321,10371,124321,9,San Diego,San Diego County Office of Education,Howard Gardner Community Charter,...,No,No,-1.303501e+07,3.847786e+06,elementary,1,POINT (-117.09551 32.64128),6956.0,06073012502,Census Tract 125.02


In [41]:
# checking schools that didn't match to a tract
schools_tract_join['GEOID'].isna().sum()

np.int64(10)

In [42]:
# looking at schools that didn't join to a tract
unmatched_schools = schools_tract_join[schools_tract_join['GEOID'].isna()].copy()

unmatched_schools[
    ['School Name', 'District Name', 'School Type', 'School Level', 'Latitude', 'Longitude']
]

,School Name,District Name,School Type,School Level,Latitude,Longitude
6896,MethodSchools,Dehesa Elementary,K-12,Other,33.556364,-117.136194
6986,Harbor Springs Charter,Julian Union Elementary,K-12,Other,33.511716,-117.157629
7038,Compass Charter Schools of San Diego,Mountain Empire Unified,K-12,Other,34.155778,-118.830763
7040,Elite Academic Academy - Mountain Empire,Mountain Empire Unified,K-12,Other,33.500301,-117.161088
7375,Insight @ San Diego,Spencer Valley Elementary,K-12,Other,34.275889,-118.798010
7377,California Virtual Academy @ San Diego,Spencer Valley Elementary,K-12,Other,34.275889,-118.798010
7502,California Pacific Charter - San Diego,Warner Unified,K-12,Other,33.690218,-117.895400
7504,Sage Oak Charter School - South,Warner Unified,K-12,Other,34.038346,-117.153726
7505,Excel Academy Charter,Warner Unified,K-12,Other,33.646987,-117.734685
7506,Pathways Academy Charter School - Adult Education,Warner Unified,High,High,33.683855,-117.205473


Some schools didn’t join to a San Diego County census tract since they're technically outside of San Diego County. I'll exclude them from the study. 

In [43]:
# keeping only schools that matched to a San Diego County tract
schools_tract_matched = schools_tract_join[schools_tract_join['GEOID'].notna()].copy()

schools_tract_matched.shape

(741, 104)

## Create tract-level school counts

Now I’m aggregating the matched schools to the tract level.

This gives each census tract school access features like total schools, elementary schools, middle schools, high schools, other schools, and charter schools.

In [44]:
# counting total schools by tract
school_counts = (
    schools_tract_matched
    .groupby('GEOID')
    .size()
    .reset_index(name='school_count'))

school_counts.head()

,GEOID,school_count
0,06073000202,1
1,06073000400,1
2,06073000600,2
3,06073001202,1
4,06073001302,1


In [45]:
# counting school levels by tract
school_level_counts = (
    schools_tract_matched
    .groupby(['GEOID', 'school_level_clean'])
    .size()
    .reset_index(name='count'))

school_level_counts.head()

,GEOID,school_level_clean,count
0,06073000202,elementary,1
1,06073000400,elementary,1
2,06073000600,elementary,1
3,06073000600,other,1
4,06073001202,elementary,1


In [46]:
# turning school level counts into separate columns
school_level_pivot = (
    school_level_counts
    .pivot_table(
        index='GEOID',
        columns='school_level_clean',
        values='count',
        fill_value=0)
    .reset_index())

school_level_pivot.head()

school_level_clean,GEOID,elementary,high,middle,other
0,06073000202,1.0,0.0,0.0,0.0
1,06073000400,1.0,0.0,0.0,0.0
2,06073000600,1.0,0.0,0.0,1.0
3,06073001202,1.0,0.0,0.0,0.0
4,06073001302,0.0,0.0,0.0,1.0


In [47]:
# renaming school level columns
school_level_pivot = school_level_pivot.rename(columns={
    'elementary': 'elementary_school_count',
    'middle': 'middle_school_count',
    'high': 'high_school_count',
    'other': 'other_school_count',
    'unknown': 'unknown_school_count'})

school_level_pivot.head()

school_level_clean,GEOID,elementary_school_count,high_school_count,middle_school_count,other_school_count
0,06073000202,1.0,0.0,0.0,0.0
1,06073000400,1.0,0.0,0.0,0.0
2,06073000600,1.0,0.0,0.0,1.0
3,06073001202,1.0,0.0,0.0,0.0
4,06073001302,0.0,0.0,0.0,1.0


In [48]:
# counting charter schools by tract
charter_counts = (
    schools_tract_matched
    .groupby('GEOID')['charter_flag']
    .sum()
    .reset_index(name='charter_school_count'))

charter_counts.head()

,GEOID,charter_school_count
0,06073000202,0
1,06073000400,0
2,06073000600,0
3,06073001202,0
4,06073001302,0


## Check tract and district alignment

Census tracts and school district boundaries don't always match perfectly.

To connect tracts to districts, I’m checking how much each tract overlaps each school district. Then I’ll assign each tract to the district with the largest overlap. This gives a better district match than assuming the boundaries line up exactly.

In [49]:
# projecting to a California CRS before calculating areas
sd_tracts_area = sd_tracts.to_crs('EPSG:3310')
districts_area = districts.to_crs('EPSG:3310')

In [50]:
# finding where tracts and district boundaries overlap
tract_district_overlap = gpd.overlay(
    sd_tracts_area,
    districts_area,
    how='intersection')

tract_district_overlap.head()

,GEOID,NAMELSAD,OBJECTID,Year,FedID,CDCode,CDSCode,CountyName,DistrictNa,DistrictTy,...,MIGcount,MIGpct,SWDcount,SWDpct,SEDcount,SEDpct,DistrctAre,Shape__Are,Shape__Len,geometry
0,06073008331,Census Tract 83.31,557,2024-25,0610740,3768056,37680560000000,San Diego,Del Mar Union Elementary,Elementary,...,0,0.0,368,10.5,361,10.3,15.583487,5.772043e+07,42960.055469,"POLYGON ((259221.387 -559657.648, 259221.436 -..."
1,06073008331,Census Tract 83.31,576,2024-25,0634380,3768346,37683460000000,San Diego,San Dieguito Union High,High,...,19,0.2,1372,11.4,2158,17.9,81.722589,3.052661e+08,88709.779222,"POLYGON ((259221.387 -559657.648, 259221.436 -..."
2,06073008331,Census Tract 83.31,580,2024-25,0636990,3768387,37683870000000,San Diego,Solana Beach Elementary,Elementary,...,0,0.0,287,11.0,367,14.0,21.134323,7.846482e+07,62140.687005,"POLYGON ((259578.062 -559081.797, 259597.205 -..."
3,06073008336,Census Tract 83.36,572,2024-25,0631530,3768296,37682960000000,San Diego,Poway Unified,Unified,...,5,0.0,5317,15.5,5447,15.8,99.812420,3.699967e+08,109096.246347,"POLYGON ((267821.309 -556373.822, 267821.733 -..."
4,06073008337,Census Tract 83.37,572,2024-25,0631530,3768296,37682960000000,San Diego,Poway Unified,Unified,...,5,0.0,5317,15.5,5447,15.8,99.812420,3.699967e+08,109096.246347,"POLYGON ((267059.008 -557964.822, 267065.274 -..."


In [51]:
# calculating overlap area
tract_district_overlap['overlap_area'] = tract_district_overlap.geometry.area

In [52]:
# calculating full tract area
tract_areas = sd_tracts_area[['GEOID', 'geometry']].copy()
tract_areas['tract_area'] = tract_areas.geometry.area

tract_areas = tract_areas[['GEOID', 'tract_area']]

In [53]:
# adding tract area and calculating overlap percent
tract_district_overlap = tract_district_overlap.merge(
    tract_areas,
    on='GEOID',
    how='left')

tract_district_overlap['overlap_pct'] = (
    tract_district_overlap['overlap_area'] / tract_district_overlap['tract_area'])

tract_district_overlap[['GEOID', 'NAMELSAD', 'DistrictNa', 'DistrictTy', 'overlap_pct']].head()

,GEOID,NAMELSAD,DistrictNa,DistrictTy,overlap_pct
0,06073008331,Census Tract 83.31,Del Mar Union Elementary,Elementary,0.999965
1,06073008331,Census Tract 83.31,San Dieguito Union High,High,1.000000
2,06073008331,Census Tract 83.31,Solana Beach Elementary,Elementary,0.000035
3,06073008336,Census Tract 83.36,Poway Unified,Unified,1.000000
4,06073008337,Census Tract 83.37,Poway Unified,Unified,1.000000


In [54]:
# keeping the district with the largest overlap for each tract
tract_district_main = (
    tract_district_overlap
    .sort_values(['GEOID', 'overlap_pct'], ascending=[True, False])
    .drop_duplicates(subset='GEOID', keep='first')
    .copy())

tract_district_main[
    ['GEOID', 'NAMELSAD', 'DistrictNa', 'DistrictTy', 'GradeLow', 'GradeHigh', 'overlap_pct']].head()

,GEOID,NAMELSAD,DistrictNa,DistrictTy,GradeLow,GradeHigh,overlap_pct
1029,06073000100,Census Tract 1,San Diego Unified,Unified,PK,12,1.0
1391,06073000201,Census Tract 2.01,San Diego Unified,Unified,PK,12,1.0
1421,06073000202,Census Tract 2.02,San Diego Unified,Unified,PK,12,1.0
229,06073000301,Census Tract 3.01,San Diego Unified,Unified,PK,12,1.0
230,06073000302,Census Tract 3.02,San Diego Unified,Unified,PK,12,1.0


In [55]:
# checking how clean the tract-to-district match is
tract_district_main['overlap_pct'].describe()

count    737.000000
mean       0.954607
std        0.127757
min        0.000005
25%        0.999797
50%        1.000000
75%        1.000000
max        1.000000
Name: overlap_pct, dtype: float64

## Check weak tract-district matches

Most tracts matched clearly to one main district, but a few had low overlap percentages.

I’m checking those now so I don’t blindly assign a district when the boundary match looks weak or weird.

In [56]:
# checking tracts where the main district covers less than 90% of the tract
low_overlap_tracts = tract_district_main[
    tract_district_main['overlap_pct'] < 0.90].copy()

low_overlap_tracts[
    ['GEOID', 'NAMELSAD', 'DistrictNa', 'DistrictTy', 'GradeLow', 'GradeHigh', 'overlap_pct']].sort_values('overlap_pct').head(30)

,GEOID,NAMELSAD,DistrictNa,DistrictTy,GradeLow,GradeHigh,overlap_pct
535,06073990100,Census Tract 9901,San Diego Unified,Unified,PK,12,0.000005
1305,06073009902,Census Tract 99.02,San Diego Unified,Unified,PK,12,0.026306
1359,06073021600,Census Tract 216,Coronado Unified,Unified,PK,12,0.197456
1254,06073010601,Census Tract 106.01,South Bay Union,Elementary,PK,08,0.202117
208,06073018301,Census Tract 183.01,Oceanside Unified,Unified,PK,12,0.255740
224,06073005103,Census Tract 51.03,San Diego Unified,Unified,PK,12,0.393876
922,06073019208,Census Tract 192.08,San Marcos Unified,Unified,KG,12,0.399068
710,06073009504,Census Tract 95.04,Poway Unified,Unified,PK,12,0.405475
371,06073008202,Census Tract 82.02,San Diego Unified,Unified,PK,12,0.410662
595,06073010103,Census Tract 101.03,Sweetwater Union High,High,KG,12,0.443895


In [57]:
# counting weak tract-district matches
low_overlap_tracts.shape[0]

95

In [58]:
# checking overlap quality by district type
tract_district_main.groupby('DistrictTy')['overlap_pct'].describe()

,count,mean,std,min,25%,50%,75%,max
DistrictTy,,,,,,,,
Elementary,159.0,0.966327,0.112449,0.202117,1.000000,1.0,1.0,1.0
High,143.0,0.959391,0.106838,0.443895,0.999854,1.0,1.0,1.0
Unified,435.0,0.948751,0.138741,0.000005,0.998904,1.0,1.0,1.0


In [59]:
# keeping only the fields needed to link tracts to districts
tract_district_lookup = tract_district_main[[
    'GEOID', 'DistrictNa', 'DistrictTy', 'GradeLow', 'GradeHigh', 'overlap_pct']].copy()

tract_district_lookup = tract_district_lookup.rename(columns={
    'DistrictNa': 'district_name',
    'DistrictTy': 'district_type',
    'GradeLow':   'district_grade_low',
    'GradeHigh':  'district_grade_high',
    'overlap_pct': 'district_overlap_pct'})

tract_district_lookup.shape

(737, 6)

In [60]:
# starting with all San Diego tracts so tracts with no schools stay in the output
school_features = sd_tracts[['GEOID', 'NAMELSAD', 'geometry']].copy()

school_features = school_features.merge(
    school_counts,
    on='GEOID',
    how='left')

school_features = school_features.merge(
    school_level_pivot,
    on='GEOID',
    how='left')

school_features = school_features.merge(
    charter_counts,
    on='GEOID',
    how='left')

school_features = school_features.merge(
    tract_district_lookup,
    on='GEOID',
    how='left')

school_features.head()

,GEOID,NAMELSAD,geometry,school_count,elementary_school_count,high_school_count,middle_school_count,other_school_count,charter_school_count,district_name,district_type,district_grade_low,district_grade_high,district_overlap_pct
0,06073008331,Census Tract 83.31,"POLYGON ((-117.23082 32.94176, -117.23079 32.9...",1.0,1.0,0.0,0.0,0.0,0.0,San Dieguito Union High,High,07,12,1.0
1,06073008336,Census Tract 83.36,"POLYGON ((-117.13793 32.96927, -117.13792 32.9...",NaN,NaN,NaN,NaN,NaN,NaN,Poway Unified,Unified,PK,12,1.0
2,06073008337,Census Tract 83.37,"POLYGON ((-117.14678 32.95497, -117.14657 32.9...",1.0,1.0,0.0,0.0,0.0,0.0,Poway Unified,Unified,PK,12,1.0
3,06073011601,Census Tract 116.01,"POLYGON ((-117.10356 32.6672, -117.10314 32.66...",NaN,NaN,NaN,NaN,NaN,NaN,National Elementary,Elementary,PK,08,1.0
4,06073011602,Census Tract 116.02,"POLYGON ((-117.10154 32.66202, -117.10133 32.6...",2.0,1.0,1.0,0.0,0.0,0.0,Sweetwater Union High,High,KG,12,1.0


This builds one tract-level school feature table. The left joins keep every census tract, even if that tract has zero schools.

In [61]:
# filling missing count values with 0
count_cols = [
    'school_count',
    'elementary_school_count',
    'middle_school_count',
    'high_school_count',
    'other_school_count',
    'unknown_school_count',
    'charter_school_count']

for col in count_cols:
    if col in school_features.columns:
        school_features[col] = school_features[col].fillna(0).astype(int)

school_features.head()

,GEOID,NAMELSAD,geometry,school_count,elementary_school_count,high_school_count,middle_school_count,other_school_count,charter_school_count,district_name,district_type,district_grade_low,district_grade_high,district_overlap_pct
0,06073008331,Census Tract 83.31,"POLYGON ((-117.23082 32.94176, -117.23079 32.9...",1,1,0,0,0,0,San Dieguito Union High,High,07,12,1.0
1,06073008336,Census Tract 83.36,"POLYGON ((-117.13793 32.96927, -117.13792 32.9...",0,0,0,0,0,0,Poway Unified,Unified,PK,12,1.0
2,06073008337,Census Tract 83.37,"POLYGON ((-117.14678 32.95497, -117.14657 32.9...",1,1,0,0,0,0,Poway Unified,Unified,PK,12,1.0
3,06073011601,Census Tract 116.01,"POLYGON ((-117.10356 32.6672, -117.10314 32.66...",0,0,0,0,0,0,National Elementary,Elementary,PK,08,1.0
4,06073011602,Census Tract 116.02,"POLYGON ((-117.10154 32.66202, -117.10133 32.6...",2,1,1,0,0,0,Sweetwater Union High,High,KG,12,1.0


In [62]:
# projecting tracts before calculating area
school_features_area = school_features.to_crs('EPSG:3310')

# calculating tract area in square miles
school_features_area['tract_area_sq_mile'] = (
    school_features_area.geometry.area / 2_589_988.110336)

# calculating school density by tract
school_features_area['schools_per_sq_mile'] = (
    school_features_area['school_count'] / school_features_area['tract_area_sq_mile'])

school_features_area[['GEOID', 'NAMELSAD', 'school_count', 'tract_area_sq_mile', 'schools_per_sq_mile']].head()

,GEOID,NAMELSAD,school_count,tract_area_sq_mile,schools_per_sq_mile
0,06073008331,Census Tract 83.31,1,0.368422,2.714281
1,06073008336,Census Tract 83.36,0,0.319910,0.000000
2,06073008337,Census Tract 83.37,1,0.607183,1.646951
3,06073011601,Census Tract 116.01,0,0.283289,0.000000
4,06073011602,Census Tract 116.02,2,0.393701,5.080003


In [63]:
# adding area and school density back to the main school features table
school_features['tract_area_sq_mile'] = school_features_area['tract_area_sq_mile']
school_features['schools_per_sq_mile'] = school_features_area['schools_per_sq_mile']

school_features[
    ['GEOID', 'NAMELSAD', 'school_count', 'tract_area_sq_mile', 'schools_per_sq_mile']].head()

,GEOID,NAMELSAD,school_count,tract_area_sq_mile,schools_per_sq_mile
0,06073008331,Census Tract 83.31,1,0.368422,2.714281
1,06073008336,Census Tract 83.36,0,0.319910,0.000000
2,06073008337,Census Tract 83.37,1,0.607183,1.646951
3,06073011601,Census Tract 116.01,0,0.283289,0.000000
4,06073011602,Census Tract 116.02,2,0.393701,5.080003


In [64]:
# checking final school access columns
school_features[
    [
        'GEOID',
        'NAMELSAD',
        'school_count',
        'elementary_school_count',
        'middle_school_count',
        'high_school_count',
        'other_school_count',
        'charter_school_count',
        'tract_area_sq_mile',
        'schools_per_sq_mile',
        'district_name',
        'district_type',
        'district_overlap_pct']].head()

,GEOID,NAMELSAD,school_count,elementary_school_count,middle_school_count,high_school_count,other_school_count,charter_school_count,tract_area_sq_mile,schools_per_sq_mile,district_name,district_type,district_overlap_pct
0,06073008331,Census Tract 83.31,1,1,0,0,0,0,0.368422,2.714281,San Dieguito Union High,High,1.0
1,06073008336,Census Tract 83.36,0,0,0,0,0,0,0.319910,0.000000,Poway Unified,Unified,1.0
2,06073008337,Census Tract 83.37,1,1,0,0,0,0,0.607183,1.646951,Poway Unified,Unified,1.0
3,06073011601,Census Tract 116.01,0,0,0,0,0,0,0.283289,0.000000,National Elementary,Elementary,1.0
4,06073011602,Census Tract 116.02,2,1,0,1,0,0,0.393701,5.080003,Sweetwater Union High,High,1.0


In [65]:
# checking school access features
school_features[
    [
        'school_count',
        'elementary_school_count',
        'middle_school_count',
        'high_school_count',
        'other_school_count',
        'charter_school_count',
        'tract_area_sq_mile',
        'schools_per_sq_mile',
        'district_overlap_pct'
    ]
].describe()

,school_count,elementary_school_count,middle_school_count,high_school_count,other_school_count,charter_school_count,tract_area_sq_mile,schools_per_sq_mile,district_overlap_pct
count,737.000000,737.000000,737.000000,737.000000,737.000000,737.000000,737.000000,737.000000,737.000000
mean,1.005427,0.616011,0.138399,0.185889,0.065129,0.156038,6.140743,1.407474,0.954607
std,1.115588,0.715066,0.360938,0.482764,0.277982,0.509496,38.847567,2.095760,0.127757
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.085695,0.000000,0.000005
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.417008,0.000000,0.999797
50%,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.727699,0.687572,1.000000
75%,2.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.457910,2.012978,1.000000
max,7.000000,5.000000,2.000000,4.000000,2.000000,7.000000,631.539108,21.742177,1.000000


Confirmed that tract-level school access layer is clean before we add academic performance data.

## Academic indicators

The school access features show where schools are, then the academic indicator files give some performance metrics for districts and schools. I’m starting with ELA and math indicators, then I’ll decide whether to use district or school data.

In [66]:
# checking what record types are in the academic files
print('ELA record types:')
print(ela['rtype'].value_counts(dropna=False))

print('\nMath record types:')
print(math['rtype'].value_counts(dropna=False))

ELA record types:
rtype
S    159097
D     16971
X        20
Name: count, dtype: int64

Math record types:
rtype
S    159257
D     16983
X        20
Name: count, dtype: int64


In [67]:
# filtering to San Diego district-level records
ela_district = ela[
    (ela['rtype'] == 'D') &
    (ela['countyname'] == 'San Diego')
].copy()

math_district = math[
    (math['rtype'] == 'D') &
    (math['countyname'] == 'San Diego')
].copy()

print('ELA district:', ela_district.shape)
print('Math district:', math_district.shape)

ELA district: (823, 37)
Math district: (825, 37)


In [68]:
# previewing district academic records
ela_district.head()

,cds,rtype,schoolname,districtname,countyname,charter_flag,coe_flag,dass_flag,studentgroup,currdenom,...,currdenom_withoutPRLOSS,currstatus_withoutPRLOSS,priorprate_enrolled,priorprate_tested,priorprate,priornumPRLOSS,priordenom_withoutPRLOSS,priorstatus_withoutPRLOSS,indicator,reportingyear
118434,37103710000000,D,NaN,San Diego County Office of Education,San Diego,NaN,Y,NaN,AA,15,...,15,-142.6,41.0,34.0,83.0,5.0,14,-123.9,ELA,2025
118435,37103710000000,D,NaN,San Diego County Office of Education,San Diego,NaN,Y,NaN,AI,2,...,2,NaN,NaN,NaN,NaN,NaN,0,NaN,ELA,2025
118436,37103710000000,D,NaN,San Diego County Office of Education,San Diego,NaN,Y,NaN,ALL,236,...,222,-116.6,369.0,316.0,86.0,33.0,172,-127.1,ELA,2025
118437,37103710000000,D,NaN,San Diego County Office of Education,San Diego,NaN,Y,NaN,EL,84,...,82,-145.5,136.0,120.0,88.0,9.0,67,-149.1,ELA,2025
118438,37103710000000,D,NaN,San Diego County Office of Education,San Diego,NaN,Y,NaN,ELO,69,...,69,-166.0,119.0,106.0,89.0,7.0,62,-150.2,ELA,2025


In [69]:
# previewing math district academic records
math_district.head()

,cds,rtype,schoolname,districtname,countyname,charter_flag,coe_flag,dass_flag,studentgroup,currdenom,...,currdenom_withoutPRLOSS,currstatus_withoutPRLOSS,priorprate_enrolled,priorprate_tested,priorprate,priornumPRLOSS,priordenom_withoutPRLOSS,priorstatus_withoutPRLOSS,indicator,reportingyear
118537,37103710000000,D,NaN,San Diego County Office of Education,San Diego,NaN,Y,NaN,AA,15,...,15,-206.5,41.0,34.0,83.0,5.0,15,-168.1,MATH,2025
118538,37103710000000,D,NaN,San Diego County Office of Education,San Diego,NaN,Y,NaN,AI,2,...,2,NaN,NaN,NaN,NaN,NaN,0,NaN,MATH,2025
118539,37103710000000,D,NaN,San Diego County Office of Education,San Diego,NaN,Y,NaN,ALL,244,...,224,-167.1,371.0,312.0,84.0,39.0,173,-175.7,MATH,2025
118540,37103710000000,D,NaN,San Diego County Office of Education,San Diego,NaN,Y,NaN,EL,89,...,86,-180.2,142.0,125.0,88.0,10.0,70,-189.2,MATH,2025
118541,37103710000000,D,NaN,San Diego County Office of Education,San Diego,NaN,Y,NaN,ELO,75,...,73,-200.8,127.0,113.0,89.0,8.0,65,-188.2,MATH,2025


In [70]:
# checking useful academic fields
academic_cols = [
    'cds',
    'schoolname',
    'districtname',
    'countyname',
    'studentgroup',
    'currdenom',
    'currstatus',
    'statuslevel',
    'change',
    'changelevel',
    'indicator',
    'reportingyear']

ela_district[academic_cols].head()

,cds,schoolname,districtname,countyname,studentgroup,currdenom,currstatus,statuslevel,change,changelevel,indicator,reportingyear
118434,37103710000000,NaN,San Diego County Office of Education,San Diego,AA,15,-142.6,1,36.3,5,ELA,2025
118435,37103710000000,NaN,San Diego County Office of Education,San Diego,AI,2,NaN,0,NaN,0,ELA,2025
118436,37103710000000,NaN,San Diego County Office of Education,San Diego,ALL,236,-129.4,1,30.8,5,ELA,2025
118437,37103710000000,NaN,San Diego County Office of Education,San Diego,EL,84,-150.0,1,20.9,5,ELA,2025
118438,37103710000000,NaN,San Diego County Office of Education,San Diego,ELO,69,-166.0,1,2.8,3,ELA,2025


In [71]:
# keeping overall district results only
ela_district_all = ela_district[ela_district['studentgroup'] == 'ALL'].copy()
math_district_all = math_district[math_district['studentgroup'] == 'ALL'].copy()

print('ELA district all:', ela_district_all.shape)
print('Math district all:', math_district_all.shape)

ELA district all: (44, 37)
Math district all: (44, 37)


In [72]:
# checking overall district academic records
ela_district_all[academic_cols].head()

,cds,schoolname,districtname,countyname,studentgroup,currdenom,currstatus,statuslevel,change,changelevel,indicator,reportingyear
118436,37103710000000,NaN,San Diego County Office of Education,San Diego,ALL,236,-129.4,1,30.8,5,ELA,2025
118734,37679670000000,NaN,Alpine Union Elementary,San Diego,ALL,847,-8.7,2,5.5,4,ELA,2025
118814,37679830000000,NaN,Borrego Springs Unified,San Diego,ALL,166,-46.7,2,13.8,4,ELA,2025
118886,37679910000000,NaN,Cajon Valley Union,San Diego,ALL,9545,-51.1,2,2.4,3,ELA,2025
119415,37680070000000,NaN,Cardiff Elementary,San Diego,ALL,373,54.2,5,6.5,4,ELA,2025


In [73]:
# keeping only the useful academic fields
ela_clean = ela_district_all[
    ['districtname', 'currdenom', 'currstatus', 'statuslevel', 'change', 'changelevel']].copy()

math_clean = math_district_all[
    ['districtname', 'currdenom', 'currstatus', 'statuslevel', 'change', 'changelevel']].copy()

In [74]:
# renaming columns so ELA and math don't overlap later
ela_clean = ela_clean.rename(columns={
    'districtname': 'district_name',
    'currdenom': 'ela_students_tested',
    'currstatus': 'ela_current_status',
    'statuslevel': 'ela_status_level',
    'change': 'ela_change',
    'changelevel': 'ela_change_level'})

math_clean = math_clean.rename(columns={
    'districtname': 'district_name',
    'currdenom': 'math_students_tested',
    'currstatus': 'math_current_status',
    'statuslevel': 'math_status_level',
    'change': 'math_change',
    'changelevel': 'math_change_level'})

In [75]:
# checking cleaned academic tables
ela_clean.head()

,district_name,ela_students_tested,ela_current_status,ela_status_level,ela_change,ela_change_level
118436,San Diego County Office of Education,236,-129.4,1,30.8,5
118734,Alpine Union Elementary,847,-8.7,2,5.5,4
118814,Borrego Springs Unified,166,-46.7,2,13.8,4
118886,Cajon Valley Union,9545,-51.1,2,2.4,3
119415,Cardiff Elementary,373,54.2,5,6.5,4


In [76]:
math_clean.head()

,district_name,math_students_tested,math_current_status,math_status_level,math_change,math_change_level
118539,San Diego County Office of Education,244,-181.9,1,25.5,5
118837,Alpine Union Elementary,845,-22.7,3,10.6,4
118917,Borrego Springs Unified,167,-85.4,2,15.9,5
118989,Cajon Valley Union,9748,-82.4,2,-1.4,3
119518,Cardiff Elementary,374,40.4,5,4.8,4


Creates one clean row per district for ELA and math, with clear column names before merging them together.

In [77]:
# merging ELA and math district indicators
academic_district = ela_clean.merge(
    math_clean,
    on='district_name',
    how='outer')

academic_district.head()

,district_name,ela_students_tested,ela_current_status,ela_status_level,ela_change,ela_change_level,math_students_tested,math_current_status,math_status_level,math_change,math_change_level
0,Alpine Union Elementary,847,-8.7,2,5.5,4,845,-22.7,3,10.6,4
1,Bonsall Unified,1209,7.4,3,7.5,4,1208,-27.9,2,1.5,3
2,Borrego Springs Unified,166,-46.7,2,13.8,4,167,-85.4,2,15.9,5
3,Cajon Valley Union,9545,-51.1,2,2.4,3,9748,-82.4,2,-1.4,3
4,Cardiff Elementary,373,54.2,5,6.5,4,374,40.4,5,4.8,4


In [78]:
# checking merged academic table
print(academic_district.shape)

academic_district.isna().sum().sort_values(ascending=False)

(44, 11)


district_name           0
ela_students_tested     0
ela_current_status      0
ela_status_level        0
ela_change              0
ela_change_level        0
math_students_tested    0
math_current_status     0
math_status_level       0
math_change             0
math_change_level       0
dtype: int64

In [79]:
# creating a simple academic strength score from ELA and math status levels
academic_district['academic_strength_score'] = (
    academic_district['ela_status_level'] + academic_district['math_status_level']) / 2

academic_district[
    [
        'district_name',
        'ela_status_level',
        'math_status_level',
        'academic_strength_score']].sort_values('academic_strength_score', ascending=False).head(10)

,district_name,ela_status_level,math_status_level,academic_strength_score
9,Del Mar Union Elementary,5,5,5.0
4,Cardiff Elementary,5,5,5.0
27,Rancho Santa Fe Elementary,5,5,5.0
25,Poway Unified,5,5,5.0
31,San Dieguito Union High,5,5,5.0
36,Solana Beach Elementary,5,5,5.0
5,Carlsbad Unified,5,4,4.5
7,Coronado Unified,5,4,4.5
10,Encinitas Union Elementary,5,4,4.5
33,San Pasqual Union Elementary,4,4,4.0


In [80]:
# grouping academic strength into simple tiers
def academic_tier(score):
    if score >= 4.5:
        return 'high'
    elif score >= 3.5:
        return 'medium_high'
    elif score >= 2.5:
        return 'medium'
    else:
        return 'lower'


academic_district['academic_strength_tier'] = academic_district['academic_strength_score'].apply(academic_tier)

In [81]:
# checking academic strength tiers
academic_district['academic_strength_tier'].value_counts()

academic_strength_tier
lower          22
high            9
medium_high     7
medium          6
Name: count, dtype: int64

In [82]:
# previewing academic district scores
academic_district[
    [
        'district_name',
        'ela_status_level',
        'math_status_level',
        'academic_strength_score',
        'academic_strength_tier'
    ]
].sort_values('academic_strength_score', ascending=False).head(15)

,district_name,ela_status_level,math_status_level,academic_strength_score,academic_strength_tier
9,Del Mar Union Elementary,5,5,5.0,high
4,Cardiff Elementary,5,5,5.0,high
27,Rancho Santa Fe Elementary,5,5,5.0,high
25,Poway Unified,5,5,5.0,high
31,San Dieguito Union High,5,5,5.0,high
36,Solana Beach Elementary,5,5,5.0,high
5,Carlsbad Unified,5,4,4.5,high
7,Coronado Unified,5,4,4.5,high
10,Encinitas Union Elementary,5,4,4.5,high
33,San Pasqual Union Elementary,4,4,4.0,medium_high


Need to connect each tract’s dominant district to the ELA/math academic strength score. After this, each tract has school access and school strength fields.

In [83]:
# adding academic district scores to tract-level school features
school_features = school_features.merge(
    academic_district,
    on='district_name',
    how='left'
)

school_features.head()

,GEOID,NAMELSAD,geometry,school_count,elementary_school_count,high_school_count,middle_school_count,other_school_count,charter_school_count,district_name,...,ela_status_level,ela_change,ela_change_level,math_students_tested,math_current_status,math_status_level,math_change,math_change_level,academic_strength_score,academic_strength_tier
0,06073008331,Census Tract 83.31,"POLYGON ((-117.23082 32.94176, -117.23079 32.9...",1,1,0,0,0,0,San Dieguito Union High,...,5,8.2,4,5473,56.8,5,5.8,4,5.0,high
1,06073008336,Census Tract 83.36,"POLYGON ((-117.13793 32.96927, -117.13792 32.9...",0,0,0,0,0,0,Poway Unified,...,5,0.8,3,17419,39.8,5,2.6,3,5.0,high
2,06073008337,Census Tract 83.37,"POLYGON ((-117.14678 32.95497, -117.14657 32.9...",1,1,0,0,0,0,Poway Unified,...,5,0.8,3,17419,39.8,5,2.6,3,5.0,high
3,06073011601,Census Tract 116.01,"POLYGON ((-117.10356 32.6672, -117.10314 32.66...",0,0,0,0,0,0,National Elementary,...,2,7.2,4,2152,-54.5,2,8.2,4,2.0,lower
4,06073011602,Census Tract 116.02,"POLYGON ((-117.10154 32.66202, -117.10133 32.6...",2,1,1,0,0,0,Sweetwater Union High,...,2,4.8,4,14254,-62.9,2,9.8,4,2.0,lower


In [84]:
# checking missing academic scores after merge
school_features[
    [
        'district_name',
        'academic_strength_score',
        'academic_strength_tier']].isna().sum()

district_name              0
academic_strength_score    0
academic_strength_tier     0
dtype: int64

In [85]:
# previewing tract-level school access and academic fields
school_features[
    [
        'GEOID',
        'NAMELSAD',
        'district_name',
        'district_type',
        'school_count',
        'schools_per_sq_mile',
        'ela_status_level',
        'math_status_level',
        'academic_strength_score',
        'academic_strength_tier'
    ]
].head()

,GEOID,NAMELSAD,district_name,district_type,school_count,schools_per_sq_mile,ela_status_level,math_status_level,academic_strength_score,academic_strength_tier
0,06073008331,Census Tract 83.31,San Dieguito Union High,High,1,2.714281,5,5,5.0,high
1,06073008336,Census Tract 83.36,Poway Unified,Unified,0,0.000000,5,5,5.0,high
2,06073008337,Census Tract 83.37,Poway Unified,Unified,1,1.646951,5,5,5.0,high
3,06073011601,Census Tract 116.01,National Elementary,Elementary,0,0.000000,2,2,2.0,lower
4,06073011602,Census Tract 116.02,Sweetwater Union High,High,2,5.080003,2,2,2.0,lower


In [86]:
# checking academic strength score distribution
school_features['academic_strength_score'].describe()

count    737.000000
mean       3.130936
std        1.013573
min        1.500000
25%        2.000000
50%        3.500000
75%        3.500000
max        5.000000
Name: academic_strength_score, dtype: float64

In [87]:
# counting tracts by academic strength tier
school_features['academic_strength_tier'].value_counts()

academic_strength_tier
medium_high    338
lower          274
high           115
medium          10
Name: count, dtype: int64

In [88]:
# looking at tracts with the strongest academic district scores
school_features[
    [
        'GEOID',
        'NAMELSAD',
        'district_name',
        'district_type',
        'school_count',
        'schools_per_sq_mile',
        'academic_strength_score',
        'academic_strength_tier'
    ]
].sort_values('academic_strength_score', ascending=False).head(15)

,GEOID,NAMELSAD,district_name,district_type,school_count,schools_per_sq_mile,academic_strength_score,academic_strength_tier
719,06073008366,Census Tract 83.66,Poway Unified,Unified,1,0.429218,5.0,high
665,06073017048,Census Tract 170.48,Poway Unified,Unified,1,0.684759,5.0,high
652,06073008330,Census Tract 83.30,San Dieguito Union High,High,1,1.688957,5.0,high
651,06073008327,Census Tract 83.27,San Dieguito Union High,High,2,0.819076,5.0,high
569,06073008365,Census Tract 83.65,Poway Unified,Unified,0,0.000000,5.0,high
562,06073017502,Census Tract 175.02,San Dieguito Union High,High,4,5.429031,5.0,high
561,06073017501,Census Tract 175.01,San Dieguito Union High,High,0,0.000000,5.0,high
559,06073017040,Census Tract 170.40,Poway Unified,Unified,1,0.877262,5.0,high
558,06073017039,Census Tract 170.39,Poway Unified,Unified,2,1.080762,5.0,high
557,06073017037,Census Tract 170.37,Poway Unified,Unified,1,0.814473,5.0,high


In [89]:
# looking at tracts with the weakest academic districts
school_features[
    [
        'GEOID',
        'NAMELSAD',
        'district_name',
        'district_type',
        'school_count',
        'schools_per_sq_mile',
        'academic_strength_score',
        'academic_strength_tier'
    ]
].sort_values('academic_strength_score').head(15)

,GEOID,NAMELSAD,district_name,district_type,school_count,schools_per_sq_mile,academic_strength_score,academic_strength_tier
672,06073021202,Census Tract 212.02,Mountain Empire Unified,Unified,1,0.014384,1.5,lower
198,06073019111,Census Tract 191.11,Valley Center-Pauma Unified,Unified,4,0.356615,1.5,lower
199,06073019110,Census Tract 191.10,Valley Center-Pauma Unified,Unified,2,0.096717,1.5,lower
211,06073019108,Census Tract 191.08,Valley Center-Pauma Unified,Unified,1,0.013077,1.5,lower
183,06073021101,Census Tract 211.01,Mountain Empire Unified,Unified,4,0.017043,1.5,lower
195,06073021102,Census Tract 211.02,Mountain Empire Unified,Unified,1,0.004597,1.5,lower
387,06073019107,Census Tract 191.07,Valley Center-Pauma Unified,Unified,0,0.000000,1.5,lower
483,06073019103,Census Tract 191.03,Valley Center-Pauma Unified,Unified,3,0.123431,1.5,lower
142,06073020711,Census Tract 207.11,Valley Center-Pauma Unified,Unified,2,0.051024,1.5,lower
504,06073019105,Census Tract 191.05,Valley Center-Pauma Unified,Unified,0,0.000000,1.5,lower


## Final school features

The school access features show how many schools are located in each tract.

The academic strength score uses district-level ELA and math status levels. This doesn't fully measure school quality, but it gives a simple public-data signal for school district performance.

In [90]:
# keeping final school feature columns
school_features_final = school_features[
    [
        'GEOID',
        'NAMELSAD',
        'school_count',
        'elementary_school_count',
        'middle_school_count',
        'high_school_count',
        'other_school_count',
        'charter_school_count',
        'tract_area_sq_mile',
        'schools_per_sq_mile',
        'district_name',
        'district_type',
        'district_grade_low',
        'district_grade_high',
        'district_overlap_pct',
        'ela_students_tested',
        'ela_current_status',
        'ela_status_level',
        'ela_change',
        'ela_change_level',
        'math_students_tested',
        'math_current_status',
        'math_status_level',
        'math_change',
        'math_change_level',
        'academic_strength_score',
        'academic_strength_tier'
    ]
].copy()

school_features_final.head()

,GEOID,NAMELSAD,school_count,elementary_school_count,middle_school_count,high_school_count,other_school_count,charter_school_count,tract_area_sq_mile,schools_per_sq_mile,...,ela_status_level,ela_change,ela_change_level,math_students_tested,math_current_status,math_status_level,math_change,math_change_level,academic_strength_score,academic_strength_tier
0,06073008331,Census Tract 83.31,1,1,0,0,0,0,0.368422,2.714281,...,5,8.2,4,5473,56.8,5,5.8,4,5.0,high
1,06073008336,Census Tract 83.36,0,0,0,0,0,0,0.319910,0.000000,...,5,0.8,3,17419,39.8,5,2.6,3,5.0,high
2,06073008337,Census Tract 83.37,1,1,0,0,0,0,0.607183,1.646951,...,5,0.8,3,17419,39.8,5,2.6,3,5.0,high
3,06073011601,Census Tract 116.01,0,0,0,0,0,0,0.283289,0.000000,...,2,7.2,4,2152,-54.5,2,8.2,4,2.0,lower
4,06073011602,Census Tract 116.02,2,1,0,1,0,0,0.393701,5.080003,...,2,4.8,4,14254,-62.9,2,9.8,4,2.0,lower


In [91]:
# checking final school feature columns
school_features_final.columns.tolist()

['GEOID',
 'NAMELSAD',
 'school_count',
 'elementary_school_count',
 'middle_school_count',
 'high_school_count',
 'other_school_count',
 'charter_school_count',
 'tract_area_sq_mile',
 'schools_per_sq_mile',
 'district_name',
 'district_type',
 'district_grade_low',
 'district_grade_high',
 'district_overlap_pct',
 'ela_students_tested',
 'ela_current_status',
 'ela_status_level',
 'ela_change',
 'ela_change_level',
 'math_students_tested',
 'math_current_status',
 'math_status_level',
 'math_change',
 'math_change_level',
 'academic_strength_score',
 'academic_strength_tier']

I kept columns that describe school access, district context, and academic strength.

- Count fields show how many schools are located in each tract.
- District fields show the dominant overlapping school district for each tract.
- ELA/math fields give a simple academic signal, not a perfect measure of school quality.

In [96]:
school_features_final['schools_per_sq_mile'] = school_features_final['schools_per_sq_mile'].fillna(0)

In [92]:
# saving final school features
output_path = Path('../data/processed/school_features_by_tract.csv')

school_features_final.to_csv(output_path, index=False)

output_path

WindowsPath('../data/processed/school_features_by_tract.csv')

In [94]:
school_features_final.shape

(737, 27)

In [95]:
count_cols = ['school_count', 'elementary_school_count', 'middle_school_count',
              'high_school_count', 'other_school_count', 'charter_school_count']
school_features_final[count_cols] = school_features_final[count_cols].fillna(0).astype(int)

In [97]:
# names in tract lookup not found in academic data
set(school_features['district_name'].dropna()) - set(academic_district['district_name'].dropna())

set()